# CINEOS Mission One — T4 reliable renderer

CogVideoX-2B is loaded with sequential CPU offload. Every shot is decoded and visually validated before assembly. Diagnostics remain downloadable after failure.


In [ ]:
# Pinned, tested-together runtime (no unconstrained upgrades).
%pip install -q diffusers==0.30.3 transformers==4.44.2 accelerate==0.34.2 sentencepiece==0.2.0 "imageio[ffmpeg]==2.35.1" safetensors==0.4.5
!apt-get -qq update && apt-get -qq install -y ffmpeg


In [ ]:
import gc, importlib.metadata, io, json, pathlib, subprocess, time, zipfile
import torch
from google.colab import files

MISSION_ONE_SMOKE_MODE = False  # True renders and validates shot 1 only.
MIN_FREE_VRAM_GB = 2.0

def preflight(retry=True):
    if not torch.cuda.is_available(): raise RuntimeError("Mission One requires CUDA")
    free, total = torch.cuda.mem_get_info()
    if free / 2**30 < MIN_FREE_VRAM_GB and retry:
        gc.collect(); torch.cuda.empty_cache(); return preflight(False)
    if free / 2**30 < MIN_FREE_VRAM_GB:
        raise RuntimeError(f"Critically low free VRAM: {free / 2**30:.2f} GiB")
    name = torch.cuda.get_device_name(0)
    compatible = ("t4", "a10", "a100", "l4", "v100", "h100", "rtx")
    if not any(item in name.lower() for item in compatible):
        raise RuntimeError(f"Unsupported GPU: {name}")
    return {"gpu": name, "free_vram_gb": round(free/2**30, 2), "total_vram_gb": round(total/2**30, 2), "torch": torch.__version__, "diffusers": importlib.metadata.version("diffusers"), "cuda_runtime": torch.version.cuda}

hardware = preflight(); print(json.dumps(hardware, indent=2))


In [ ]:
uploaded = files.upload()
archive = next(iter(uploaded)); root = pathlib.Path("/content/mission-one"); root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[archive])) as value: value.extractall(root)
package = json.loads((root / "package.json").read_text())
shots = package["ordered_shot_packages"][:1] if MISSION_ONE_SMOKE_MODE else package["ordered_shot_packages"]
print(f"Mode: {'SMOKE (shot 1 only)' if MISSION_ONE_SMOKE_MODE else 'FULL'}, shots: {len(shots)}")


In [ ]:
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

pipe = CogVideoXPipeline.from_pretrained(
    package["model_id"], torch_dtype=torch.float16, low_cpu_mem_usage=True,
)
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_tiling(); pipe.vae.enable_slicing()


In [ ]:
def validate_video(path, shot_id):
    size = path.stat().st_size if path.exists() else 0
    base = {"shot_id": shot_id, "output_file": path.name, "file_size": size}
    if size < 2048: return {**base, "success": False, "content_status": "empty_output"}
    try:
        probe = json.loads(subprocess.run(["ffprobe","-v","error","-select_streams","v:0","-count_frames","-show_entries","stream=width,height,nb_read_frames,duration:format=duration","-of","json",str(path)], check=True, capture_output=True).stdout)
        stream = probe["streams"][0]; duration = float(stream.get("duration") or probe["format"].get("duration") or 0); count = int(stream.get("nb_read_frames") or 0)
        raw = subprocess.run(["ffmpeg","-v","error","-i",str(path),"-vf","fps=3/duration,scale=64:64,format=gray","-frames:v","3","-f","rawvideo","-"],check=True,capture_output=True).stdout
        frames = [raw[i:i+4096] for i in range(0,len(raw),4096) if len(raw[i:i+4096]) == 4096]
        if duration <= 0 or count <= 0 or not frames: return {**base,"success":False,"content_status":"decode_failure","duration":duration,"frame_count":count}
        pixels = [p for frame in frames for p in frame]; mean = sum(pixels)/len(pixels); variance = sum((p-mean)**2 for p in pixels)/len(pixels)
        deltas = [sum(abs(a-b) for a,b in zip(x,y))/4096 for x,y in zip(frames,frames[1:])]
        status = "black_frame_failure" if mean < 3 and variance < 2 else ("frozen_frame_failure" if deltas and max(deltas) < .35 else "valid")
        return {**base,"success":status == "valid","content_status":status,"duration":duration,"frame_count":count,"width":int(stream["width"]),"height":int(stream["height"]),"mean_luminance":round(mean,4),"luminance_variance":round(variance,4)}
    except Exception as error:
        return {**base,"success":False,"content_status":"decode_failure","error":str(error)}


In [ ]:
results=[]; started=time.time()
for shot in shots:
    original = {"resolution":package["resolution"],"frame_count":shot["frame_count"],"inference_steps":package["inference_steps"]}
    result = None
    for attempt in range(2):  # exactly one retry, and only for black output
        gc.collect(); torch.cuda.empty_cache(); free_before = torch.cuda.mem_get_info()[0]/2**30; shot_started=time.time()
        settings = original if attempt == 0 else {**original,"resolution":"480x320","frame_count":min(33,shot["frame_count"])}
        generator = torch.Generator(device="cpu").manual_seed(shot["seed"])
        try:
            output = pipe(prompt=shot["prompt"],negative_prompt=shot["negative_prompt"],num_frames=settings["frame_count"],num_inference_steps=settings["inference_steps"],guidance_scale=package["guidance_scale"],generator=generator)
            frames=output.frames[0]; path=root/shot["expected_output"]; export_to_video(frames,str(path),fps=package["fps"]); del frames, output
            result=validate_video(path,shot["shot_id"]); result.update({"render_time":time.time()-shot_started,"seed":shot["seed"],"model_id":package["model_id"],"gpu":hardware["gpu"],"vram_profile":{"free_before_gb":round(free_before,2)},"retry_attempted":attempt == 1,"retry_reason":"black_frame_failure" if attempt else "","original_settings":original,"retry_settings":settings if attempt else {}})
        except Exception as error:
            result={"shot_id":shot["shot_id"],"output_file":shot["expected_output"],"success":False,"content_status":"render_exception","error":str(error),"retry_attempted":False,"seed":shot["seed"],"model_id":package["model_id"],"gpu":hardware["gpu"]}
        gc.collect(); torch.cuda.empty_cache()
        if result["success"] or result["content_status"] != "black_frame_failure": break
        print(f"BLACK VIDEO detected for {shot['shot_id']}; cleaning memory and retrying once")
    results.append(result)
    if not result["success"]: print(f"FAILED {shot['shot_id']}: {result['content_status']}"); break


In [ ]:
all_valid = len(results)==len(shots) and all(item["success"] and item["content_status"]=="valid" for item in results)
film = root/"final-film.mp4"
assembly_status = "smoke_mode_complete" if MISSION_ONE_SMOKE_MODE and all_valid else "blocked"
if all_valid and not MISSION_ONE_SMOKE_MODE:
    concat=root/"concat.txt"; concat.write_text("".join(f"file '{(root/s['expected_output']).resolve()}'\n" for s in shots))
    subprocess.run(["ffmpeg","-y","-f","concat","-safe","0","-i",str(concat),"-c:v","libx264","-pix_fmt","yuv420p",str(film)],check=True); assembly_status="succeeded"
render={"project_id":package["project_id"],"expected_shots":[s["shot_id"] for s in shots],"shots":results,"final_film":film.name if assembly_status=="succeeded" else "","model_id":package["model_id"],"hardware_profile":hardware,"render_time_seconds":time.time()-started}
verification={"total_shots":len(shots),"valid_shots":sum(x["success"] for x in results),"failed_shots":sum(not x["success"] for x in results),"black_frame_failures":sum(x["content_status"]=="black_frame_failure" for x in results),"frozen_frame_warnings":sum(x["content_status"]=="frozen_frame_failure" for x in results),"final_assembly_status":assembly_status,"hardware_profile":hardware,"model_profile":{"model_id":package["model_id"],"cpu_offload":"sequential"},"total_render_time":render["render_time_seconds"],"limitations":["References are preserved but not consumed","Lip-sync is approximate"],"manual_review_requirement":any(x["content_status"] in ("frozen_frame_failure","manual_review_required") for x in results)}
(root/"render-results.json").write_text(json.dumps(render,sort_keys=True,indent=2)); (root/"verification-report.json").write_text(json.dumps(verification,sort_keys=True,indent=2))
if not all_valid: print("Assembly blocked: invalid shot. Diagnostic outputs are preserved.")


In [ ]:
# Diagnostics are always downloadable; the film is offered only after success.
diagnostics=root/"mission-one-diagnostics.zip"
with zipfile.ZipFile(diagnostics,"w") as archive:
    for path in root.iterdir():
        if path.is_file() and path.name != diagnostics.name: archive.write(path,path.name)
files.download(str(diagnostics))
if assembly_status == "succeeded": files.download(str(film))
